In [ ]:
import requests
import time
import csv

# Your contact info for Wikipedia's API - required to avoid 403 blocks
HEADERS = {
    "User-Agent": "OralHistoryResearch/1.0 (mikaelaballmer@uw.edu; academic research project)"
}

def search_wikipedia(name, retries=3, delay=1.0):
    # Base URL for the Wikipedia API
    search_url = "https://en.wikipedia.org/w/api.php"
    
    # Parameters for searching Wikipedia by name
    search_params = {
        "action": "query",       # we want to query Wikipedia
        "list": "search",        # using the search feature
        "srsearch": name,        # search term is the person's name
        "format": "json",        # return results as JSON
        "srlimit": 3             # get top 3 results to compare
    }
    
    # Try the search request up to 'retries' times in case of failure
    for attempt in range(retries):
        try:
            # Make the GET request to Wikipedia API with headers and 10s timeout
            r = requests.get(search_url, params=search_params, timeout=10, headers=HEADERS)
            
            # Only proceed if we got a successful response with content
            if r.status_code == 200 and r.text.strip():
                # Parse the JSON response
                data = r.json()
                # Exit the retry loop since we succeeded
                break
            else:
                # Log the failed attempt and wait before retrying
                print(f"  Retry {attempt+1} for {name} (status {r.status_code})")
                # Wait longer each retry (1s, 2s, 3s)
                time.sleep(delay * (attempt + 1))
        except Exception as e:
            # Catch network errors and wait before retrying
            print(f"  Retry {attempt+1} for {name}: {e}")
            time.sleep(delay * (attempt + 1))
    else:
        # If all retries failed, return a low confidence result for manual review
        return None, None, "LOW - REVIEW", 0.0, "failed after retries"

    # Extract the list of search results from the response
    hits = data.get("query", {}).get("search", [])
    
    # If no results were found, return early with low confidence
    if not hits:
        return None, None, "no_results", 0.0, "no results"
    
    # Take the top search result
    top_hit = hits[0]
    # Get the Wikipedia page title
    title = top_hit['title']
    # Build the full Wikipedia URL from the title
    url = f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
    # Get the short text snippet from the search result (used later for relevance check)
    snippet = top_hit.get('snippet', '')
    
    # Parameters to fetch the page's categories (used to verify it's a real person)
    cat_params = {
        "action": "query",       # query Wikipedia
        "titles": title,         # for the page we found
        "prop": "categories",    # return its categories
        "format": "json",        # as JSON
        "cllimit": 20            # up to 20 categories
    }
    
    # Start with empty categories list in case the request fails
    categories = []
    
    # Try fetching categories up to 'retries' times
    for attempt in range(retries):
        try:
            # Make the GET request for categories
            cat_r = requests.get(search_url, params=cat_params, timeout=10, headers=HEADERS)
            
            # Only proceed if we got a successful response with content
            if cat_r.status_code == 200 and cat_r.text.strip():
                # Parse the JSON response
                cat_data = cat_r.json()
                # Navigate to the pages section of the response
                pages = cat_data.get("query", {}).get("pages", {})
                # Extract category names as lowercase strings for easier matching
                for page in pages.values():
                    categories = [c['title'].lower() for c in page.get('categories', [])]
                # Exit retry loop since we succeeded
                break
            # Wait before retrying if response was bad
            time.sleep(delay * (attempt + 1))
        except:
            # Wait before retrying if there was a network error
            time.sleep(delay * (attempt + 1))

    # --- Confidence Scoring ---
    # Start with zero confidence and build up based on checks below
    confidence = 0.0
    # Track reasons for any confidence deductions
    reasons = []

    # CHECK 1: Do the words in the search name match the Wikipedia title?
    name_words = set(name.lower().split())      # words in the input name
    title_words = set(title.lower().split())    # words in the Wikipedia title
    # Calculate what fraction of name words appear in the title
    word_overlap = len(name_words & title_words) / max(len(name_words), 1)
    # Add up to 0.5 points based on how well the names match
    confidence += word_overlap * 0.5
    # Flag if names don't fully match (e.g. "Bob Smith" vs "Robert Smith")
    if word_overlap < 1.0:
        reasons.append(f"name mismatch: '{name}' vs '{title}'")

    # CHECK 2: Do the page categories suggest this is a real person?
    # These keywords appear in categories of pages about real people
    person_indicators = ['births', 'deaths', 'living people', 'alumni',
                         'american ', 'british ', 'scientists', 'physicists',
                         'professors', 'academics', 'historians']
    # Check if any person indicators appear anywhere in the categories
    is_person = any(ind in ' '.join(categories) for ind in person_indicators)
    if is_person:
        # Add 0.3 points if the page looks like a real person
        confidence += 0.3
    else:
        reasons.append("may not be a person")

    # CHECK 3: Does the snippet suggest this is a scientist or academic?
    # These keywords are relevant to an oral history archive of scientists
    relevant_keywords = ['physicist', 'scientist', 'professor', 'researcher',
                         'astronomer', 'chemist', 'biologist', 'mathematician',
                         'engineer', 'historian', 'nobel']
    # Check if any relevant keywords appear in the search snippet
    is_relevant = any(kw in snippet.lower() for kw in relevant_keywords)
    if is_relevant:
        # Add 0.2 points if the page seems academically relevant
        confidence += 0.2
    else:
        reasons.append("may not be a scientist/academic")

    # --- Assign confidence flag based on final score ---
    if confidence >= 0.8:
        flag = "HIGH"        # very likely correct match
    elif confidence >= 0.5:
        flag = "MEDIUM"      # probably correct but worth spot-checking
    else:
        flag = "LOW - REVIEW" # likely wrong or missing, needs manual review

    # Join all reasons into a single string, or confirm all checks passed
    reason_str = "; ".join(reasons) if reasons else "all checks passed"
    
    # Return all results
    return title, url, flag, round(confidence, 2), reason_str


# --- Main loop: process every interview subject ---
results = []         # will hold all results
high, medium, low = 0, 0, 0   # counters for each confidence level

for subject in interview_subjects:
    # Get the person's name from the current record
    name = subject['name']
    
    try:
        # Call our search function for this person
        title, url, flag, confidence, reason = search_wikipedia(name)
        
        # If no Wikipedia page was found at all, set defaults
        if title is None:
            flag, confidence, reason = "LOW - REVIEW", 0.0, "no Wikipedia results found"
            title, url = "", "Not found"
        
        # Store the result for this person
        results.append({
            'id': subject['id'],
            'name': name,
            'wikipedia_title': title,
            'wikipedia_url': url,
            'confidence': confidence,
            'flag': flag,
            'notes': reason
        })
        
        # Print result with appropriate symbol based on confidence
        if flag == "HIGH":
            high += 1
            print(f"✓  HIGH     {name} → {title}")
        elif flag == "MEDIUM":
            medium += 1
            print(f"~  MEDIUM   {name} → {title} ({reason})")
        else:
            low += 1
            print(f"✗  REVIEW   {name} → {title} ({reason})")
            
    except Exception as e:
        # If anything unexpected goes wrong, log the error and flag for review
        print(f"✗  ERROR    {name}: {e}")
        results.append({
            'id': subject['id'],
            'name': name,
            'wikipedia_title': '',
            'wikipedia_url': 'Error',
            'confidence': 0.0,
            'flag': 'LOW - REVIEW',
            'notes': str(e)
        })
    
    # Wait 0.3 seconds between requests to be polite to Wikipedia's servers
    time.sleep(0.3)

# --- Save all results to a CSV file ---
output_csv = "/Users/mikaelaballmer/Documents/UW/Capstone/individual_jsons 2/wikipedia_links.csv"
with open(output_csv, 'w', newline='', encoding='utf-8') as f:
    # Define the column names
    writer = csv.DictWriter(f, fieldnames=['id', 'name', 'wikipedia_title', 'wikipedia_url', 'confidence', 'flag', 'notes'])
    # Write the header row
    writer.writeheader()
    # Write all result rows
    writer.writerows(results)

# --- Print final summary ---
print(f"\n{'='*50}")
print(f"Total processed:     {len(results)}")
print(f"HIGH confidence:     {high}")
print(f"MEDIUM confidence:   {medium}")
print(f"LOW - needs review:  {low}")
print(f"\nSaved to: {output_csv}")